In [ ]:
!pip install pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg gdcm

In [ ]:

"""
Notebook Kaggle Final - GMIC Conforme
Convertit les DICOM RSNA en PNG 16-bits + génère le PKL conforme GMIC,
puis exporte le tout en tant que Dataset Kaggle Privé.

Testé et validé avec le pipeline GMIC complet :
  crop_mammogram.py -> get_optimal_centers.py -> run_model.py
"""

import os
import csv
import random
import json
import pickle
import pydicom
import numpy as np
import cv2
from tqdm import tqdm
from kaggle.api.kaggle_api_extended import KaggleApi


# =====================================================================
# CONFIGURATION (à modifier)
# =====================================================================
os.environ["KAGGLE_API_TOKEN"] = "KGAT_af832a437db2865ba7958068cb6b65b1"
KAGGLE_USERNAME = "joshuananceyfromepi"

# Changer PART_NUMBER pour chaque exécution : 1, 2, 3, 4 ou 5
PART_NUMBER = 2  # <-- MODIFIER ICI POUR CHAQUE PART

DATASET_NAME = f"rsna-16bit-png-part2"
DATASET_TITLE = f"RSNA 16-bit PNG part2"

RANDOM_SEED = 42
SUBSET_RATIO = 0.04  # TEST : très petit subset pour valider l'upload

INPUT_DIR = "/kaggle/input/competitions/rsna-breast-cancer-detection"
CSV_PATH = os.path.join(INPUT_DIR, "train.csv")
EXPORT_DIR = "/kaggle/working/dataset_export"

os.makedirs(EXPORT_DIR, exist_ok=True)
random.seed(RANDOM_SEED)

# =====================================================================
# 1. Lire le CSV et grouper par patient
# =====================================================================
print("1. Lecture du CSV et groupement des patients...")

patients_positive, patients_negative = [], []
patient_cancer, patient_images = {}, {}
image_meta = {}

with open(CSV_PATH) as f:
    reader = csv.DictReader(f)
    for row in reader:
        pid, iid = row["patient_id"], row["image_id"]
        cancer = int(row["cancer"])
        view = row["view"]
        lat = row["laterality"]

        patient_images.setdefault(pid, []).append(iid)
        patient_cancer[pid] = max(patient_cancer.get(pid, 0), cancer)
        image_meta[f"{pid}_{iid}"] = {
            'view': view, 'lat': lat, 'cancer': cancer
        }

for pid, label in patient_cancer.items():
    if label == 1:
        patients_positive.append(pid)
    else:
        patients_negative.append(pid)

print(f"   Patients positifs: {len(patients_positive)}")
print(f"   Patients négatifs: {len(patients_negative)}")

# =====================================================================
# 2. Sélection stratifiée aléatoire
# =====================================================================
print("\n2. Sélection stratifiée...")

n_pos = round(len(patients_positive) * SUBSET_RATIO)
n_neg = round(len(patients_negative) * SUBSET_RATIO)

# Découpage par tranche séquentielle (pas de doublons entre parts)
start_pos = n_pos * (PART_NUMBER - 1)
start_neg = n_neg * (PART_NUMBER - 1)

if PART_NUMBER == 5:
    # Dernière part : prendre tout le reste
    selected_pos = patients_positive[start_pos:]
    selected_neg = patients_negative[start_neg:]
else:
    selected_pos = patients_positive[start_pos:start_pos + n_pos]
    selected_neg = patients_negative[start_neg:start_neg + n_neg]

selected_patients = set(selected_pos + selected_neg)

files_to_process = []
for pid in selected_patients:
    for iid in patient_images[pid]:
        src_path = os.path.join(INPUT_DIR, f"train_images/{pid}/{iid}.dcm")
        dest_dir = os.path.join(EXPORT_DIR, pid)
        dest_path = os.path.join(dest_dir, f"{iid}.png")
        if os.path.exists(src_path):
            files_to_process.append((src_path, dest_path, dest_dir, pid, iid))

print(f"   Part {PART_NUMBER}/5")
print(f"   Positifs: {len(selected_pos)} | Négatifs: {len(selected_neg)} | Total: {len(selected_patients)}")
print(f"   Images à convertir: {len(files_to_process)}")

# =====================================================================
# 3. Conversion DICOM -> PNG 16-bits + construction du PKL
# =====================================================================
print("\n3. Conversion DICOM -> PNG 16-bits + construction PKL...")

patients_pkl = {}
errors = 0

for src, dest, dest_dir, pid, iid in tqdm(files_to_process, desc="Conversion"):
    try:
        os.makedirs(dest_dir, exist_ok=True)
        dicom = pydicom.dcmread(src)
        img = dicom.pixel_array.astype(np.float64)

        # Inversion MONOCHROME1 (fond blanc -> fond noir)
        if dicom.PhotometricInterpretation == "MONOCHROME1":
            img = img.max() - img

        # Normalisation en 16-bits
        if img.max() > img.min():
            img = (img - img.min()) / (img.max() - img.min()) * 65535.0

        img = np.uint16(img)
        cv2.imwrite(dest, img)

        # --- Construction du PKL conforme GMIC ---
        meta = image_meta[f"{pid}_{iid}"]
        view_key = f"{meta['lat']}-{meta['view']}"  # Ex: "L-CC", "R-MLO"

        if pid not in patients_pkl:
            patients_pkl[pid] = {
                'horizontal_flip': 'NO',
                # 4 vues obligatoires
                'L-CC': [], 'L-MLO': [], 'R-CC': [], 'R-MLO': [],
                # 8 clés de segmentation (vides, RSNA n'en fournit pas)
                'L-CC_benign_seg': [], 'L-CC_malignant_seg': [],
                'L-MLO_benign_seg': [], 'L-MLO_malignant_seg': [],
                'R-CC_benign_seg': [], 'R-CC_malignant_seg': [],
                'R-MLO_benign_seg': [], 'R-MLO_malignant_seg': [],
                # Labels cancer par sein
                'cancer_label': {
                    'benign': 0, 'left_benign': 0, 'right_benign': 0,
                    'malignant': 0, 'left_malignant': 0, 'right_malignant': 0,
                    'unknown': 0
                }
            }

        # Format GMIC : "patient_id/image_id" sans extension
        short_file_path = f"{pid}/{iid}"

        # Ajouter à la bonne vue (ignorer les vues non-standard)
        if view_key in patients_pkl[pid]:
            patients_pkl[pid][view_key].append(short_file_path)

        # Labels cancer (RSNA: cancer est par SEIN, pas par patient)
        if meta['cancer'] == 1:
            patients_pkl[pid]['cancer_label']['malignant'] = 1
            if meta['lat'] == 'L':
                patients_pkl[pid]['cancer_label']['left_malignant'] = 1
            else:
                patients_pkl[pid]['cancer_label']['right_malignant'] = 1
        else:
            patients_pkl[pid]['cancer_label']['benign'] = 1
            if meta['lat'] == 'L':
                patients_pkl[pid]['cancer_label']['left_benign'] = 1
            else:
                patients_pkl[pid]['cancer_label']['right_benign'] = 1

    except Exception as e:
        errors += 1
        tqdm.write(f"Erreur {pid}/{iid}: {e}")

print(f"\n   Conversions réussies: {len(files_to_process) - errors}")
print(f"   Erreurs: {errors}")

# =====================================================================
# 4. Sauvegarde du PKL
# =====================================================================
print("\n4. Sauvegarde du fichier exam_list_before_cropping.pkl...")

pkl_path = os.path.join(EXPORT_DIR, "exam_list_before_cropping.pkl")
exam_list = list(patients_pkl.values())

with open(pkl_path, 'wb') as f:
    pickle.dump(exam_list, f)

print(f"   {len(exam_list)} patients sauvegardés dans {pkl_path}")

# Vérification rapide
views_total = {'L-CC': 0, 'L-MLO': 0, 'R-CC': 0, 'R-MLO': 0}
for exam in exam_list:
    for v in views_total:
        views_total[v] += len(exam[v])
print(f"   Images par vue: {views_total}")

cancer_count = sum(1 for e in exam_list if e['cancer_label']['malignant'] == 1)
print(f"   Patients avec cancer: {cancer_count}/{len(exam_list)}")

# =====================================================================
# 5. Vérification avant upload
# =====================================================================
print("\n5. Vérification avant upload...")

png_count = 0
total_size = 0
for root, dirs, files in os.walk(EXPORT_DIR):
    for f in files:
        fpath = os.path.join(root, f)
        total_size += os.path.getsize(fpath)
        if f.endswith('.png'):
            png_count += 1

print(f"   Fichiers PNG : {png_count}")
print(f"   Taille totale : {total_size / (1024**2):.1f} MB")
print(f"   PKL existe : {os.path.exists(pkl_path)}")

# =====================================================================
# 6. ZIP avec suppression au fur et à mesure (évite le disk full)
# =====================================================================
import zipfile
import shutil

UPLOAD_DIR = "/kaggle/working/final_upload"
os.makedirs(UPLOAD_DIR, exist_ok=True)

zip_path = os.path.join(UPLOAD_DIR, "rsna_images.zip")

# Lister tous les fichiers à zipper
all_files = []
for root, dirs, files in os.walk(EXPORT_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f"\n6. Compression en ZIP ({len(all_files)} fichiers)...")
print(f"   Chaque PNG est supprimé après ajout au ZIP pour libérer l'espace disque.")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as zf:
    for fpath in tqdm(all_files, desc="Zip + suppression"):
        arcname = os.path.relpath(fpath, EXPORT_DIR)
        zf.write(fpath, arcname)
        os.remove(fpath)  # Supprime le fichier original immédiatement

# Nettoyer les dossiers patients vides
shutil.rmtree(EXPORT_DIR, ignore_errors=True)

zip_size = os.path.getsize(zip_path)
print(f"   ZIP créé : {zip_size / (1024**2):.1f} MB")

# =====================================================================
# 7. Upload unique vers Kaggle
# =====================================================================
print(f"\n7. Upload vers Kaggle (1 seul fichier de {zip_size / (1024**2):.1f} MB)...")

dataset_metadata = {
    "title": DATASET_TITLE,
    "id": f"{KAGGLE_USERNAME}/{DATASET_NAME}",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(os.path.join(UPLOAD_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(dataset_metadata, f)

api = KaggleApi()
api.authenticate()

api.dataset_create_new(
    folder=UPLOAD_DIR,
    public=False,
    dir_mode='skip'
)

print(f"\nUpload OK ! Verifie sur : https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_NAME}")
